# S06: two applications, one shared platform

70-minute lab. Prove reuse with two applications that have different contracts. A platform provides controlled shared capabilities; a single chatbot renamed a platform does not establish reuse.

In [ ]:
from pathlib import Path
import os, sys, json, tempfile
# Run from repository root, notebooks/, or a Colab clone.
candidates=[Path.cwd(), Path.cwd().parent, Path('/content/ai-platform-architect-labs')]
ROOT=next((p for p in candidates if (p/'engine').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open the extracted repository. Colab: follow README Setup first.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–20 min | Onboard and exercise both applications.** Policy serves buyers; spend requires finance. Shared admission control, model adapter, index version, traces and quotas live in `Platform`. The service determines identity and application. Read `engine/platform.py` before changing it.

In [ ]:
ix=Index();ix.build('v1',read_data('policies.json'));db=Purchases()
platform=Platform(ix,db,client,quota=3)
print(platform.ask(BUYER,'policy','standard supplier payment terms'))
assert platform.ask(BUYER,'spend','Nova Q1 spend')['status']=='denied'
print(platform.ask(FINANCE,'spend','Nova spend in 2026-Q1?',planned_query={'quarter':'2026-Q1','supplier':'Nova'}))

**20–40 min | Tenant, cache and quota boundaries.** The same question has different policy answers in AsterWorks and Beacon. Run repeated calls and inspect cache hits. A quota is scoped by tenant, subject and application; this is a process-local call allowance, not a distributed token or spend limit.

In [ ]:
q='standard supplier payment terms'
a=platform.ask(BUYER,'policy',q);b=platform.ask(BEACON,'policy',q)
print('Aster:',a);print('Beacon:',b)
assert not b.get('cache',False)
platform.ask(BUYER,'policy',q)
assert platform.ask(BUYER,'policy',q)['status']=='quota'
assert platform.ask(BEACON,'policy',q)['status']!='quota'
print(json.dumps(platform.events,indent=2))

**40–60 min | Make a platform change.** Add per-application quota configuration with default-preserving behavior. Write an assertion proving policy traffic cannot consume the spend allowance. Add a new entitlement version and demonstrate it does not reuse the old cache entry. Decide how a real entitlement change would invalidate an already-issued identity/session. Do not accept tenant or role from a model reply.

**60–70 min | Deliver an onboarding contract.** Specify owner, task contract, model permissions, data scope, request/token budget, SLO, audit fields and release owner for a proposed third application. Identify what belongs in shared infrastructure and what remains application logic. A single-process lab does not implement enterprise IAM, a distributed gateway or distributed quotas.

In [ ]:
ix.close();db.close()